# Week 25, Day 1: Meet the data: motor cortex during reaching

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w025/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
data = np.load(dataset("churchland2012_reach.npz"))
rates = data["rates"].astype(float)     # conditions x times x neurons, spikes per second
times = data["times"]                   # ms relative to movement onset
C, T, N = rates.shape
print(f"{C} conditions, {T} time points ({times[0]} to {times[-1]} ms), {N} neurons")

### Exercise: A first look at 218 neurons

Compute each neuron's mean rate `mean_rate`, the time `peak_time` at which its condition-averaged rate is largest, and the fraction `frac_before` of neurons whose peak comes before movement onset. Then plot a histogram of mean rates and one of peak times.

In [ ]:
mean_rate = ...
psth = rates.mean(axis=0)                # times x neurons: the average over conditions
peak_time = ...
frac_before = ...
print(f"mean rates: median {np.median(mean_rate):.1f}, range {mean_rate.min():.1f} to {mean_rate.max():.1f} spikes/s")
print(f"peak times: median {np.median(peak_time):.0f} ms;  {100 * frac_before:.0f}% of neurons peak before movement onset")
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(mean_rate, 30); ax[0].set_xlabel("mean rate (spikes/s)")
ax[1].hist(peak_time, 30); ax[1].set_xlabel("peak time (ms)"); plt.show()

In [ ]:
import numpy as np
def test_mean():
    "mean_rate is each neuron's average over conditions and times"
    assert mean_rate.shape == (N,) and np.allclose(mean_rate, rates.mean(axis=(0, 1)))

def test_peak():
    "peak_time is the time of each neuron's maximum condition-averaged rate"
    psth = rates.mean(axis=0)
    assert peak_time.shape == (N,) and np.all(peak_time == times[psth.argmax(axis=0)])

def test_fraction():
    "frac_before is the fraction of neurons peaking before movement onset"
    assert abs(frac_before - np.mean(peak_time < 0)) < 1e-12

test_mean()
test_peak()
test_fraction()
print("All checks passed")

### Exercise: Who dominates the variance?

Soft-normalise the rates into `Xn`. Write `top_share(X)`, the fraction of the total variance (summed over neurons) held by the 10% of neurons with the largest variance, and compute it before (`share_raw`) and after (`share_norm`) normalisation.

In [ ]:
rng_ = rates.max(axis=(0, 1)) - rates.min(axis=(0, 1))
Xn = ...

def top_share(X):
    v = X.reshape(-1, X.shape[-1]).var(axis=0)     # variance of each neuron over conditions and times
    k = int(round(0.1 * len(v)))
    return ...

share_raw, share_norm = top_share(rates), top_share(Xn)
print(f"top 10% of neurons hold {100 * share_raw:.0f}% of the variance before normalisation, {100 * share_norm:.0f}% after")

In [ ]:
import numpy as np
def test_norm():
    "Xn divides each neuron by its range plus 5"
    rng_ = rates.max(axis=(0, 1)) - rates.min(axis=(0, 1))
    assert np.allclose(Xn, rates / (rng_ + 5))

def test_share():
    "the top 10% of neurons hold over half the variance before normalisation and under a third after"
    assert share_raw > 0.5 and share_norm < 1 / 3

test_norm()
test_share()
print("All checks passed")

### Exercise: Splitting the variance

Make `X` by subtracting the cross-condition mean from `Xn`. Then compute `frac_common`, the share of the total variance (about each neuron's grand mean) that the cross-condition mean accounts for: C times the summed squared deviations of the mean from the grand mean, divided by the total sum of squared deviations.

In [ ]:
X = ...
grand = Xn.mean(axis=(0, 1))                      # each neuron's grand mean
total = ((Xn - grand) ** 2).sum()
common = ...
frac_common = common / total
print(f"the condition-independent signal is {100 * frac_common:.0f}% of the variance; {100 * (1 - frac_common):.0f}% differs between conditions")

In [ ]:
import numpy as np
def test_X():
    "X is Xn with the cross-condition mean removed at each time point"
    assert np.allclose(X, Xn - Xn.mean(axis=0, keepdims=True)) and np.abs(X.mean(axis=0)).max() < 1e-12

def test_split():
    "frac_common is between 0 and 1 and the two parts add up to the total variance about each neuron's grand mean"
    tot = ((Xn - Xn.mean(axis=(0, 1))) ** 2).sum()
    common = C * ((Xn.mean(axis=0) - Xn.mean(axis=(0, 1))) ** 2).sum()
    assert abs(frac_common - common / tot) < 1e-9 and abs(common + (X ** 2).sum() - tot) / tot < 1e-9

test_X()
test_split()
print("All checks passed")